In [38]:
import pennylane as qml
from pennylane import numpy as np

## Quantum state preparation

For **n** qubits, the default initial state is $\otimes_{i=0}^{n} \ket{0}$

If we want to change the initial state, we need to make sure that the new initial state is normalized

For that we can use `qml.StatePrep`

In [2]:
dev = qml.device("default.qubit",wires=2)

@qml.qnode(dev)
def circuit(state=None):
    qml.StatePrep(state, wires=range(2),normalize=True)
    return qml.state()

state = circuit([1/2,1/2,1/2,1/2])

print(state)

[0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]


The previous method allows us to create an initial state in any basis

However, if we want to create an initial state in the computational basis we can do it resorting to `qml.BasisState`

This function uses `np.array` with (the state being prepared is $\ket{100}$):
- State enumeration
    ~~~
    qml.BasisState(np.array([1,0,0]), wires=range(3))
    ~~~
- Binary shape input
    ~~~
    qml.BasisState(np.array(4), wires=range(3))
    ~~~

In [3]:
dev3 = qml.device("default.qubit", wires=3)

@qml.qnode(dev3)
def circ1(state=None):
    qml.BasisState(np.array([0,0,1]), wires=range(3)) #|012>
    return qml.state()

print(circ1())

@qml.qnode(dev3)
def circ2(state=None):
    qml.BasisState(np.array(4), wires=range(3)) # 4 = 100 in binary
    return qml.state()

print(circ2())


[0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j]


#### Prepare the quantum state $\ket{\psi} = \alpha \ket{001} + \beta \ket{010} + \gamma \ket{001}$ using ´qml.StatePrep´

In [4]:
@qml.qnode(dev3)
def prep_circuit(alpha, beta, gamma):
    """
    Prepares the state alpha|001> + beta|010> + gamma|100>.
    Args:
    alpha, beta, gamma (np.complex): The coefficients of the quantum state
    to prepare.
    Returns:
    (np.array): The quantum state
    """

    st=[0,alpha,beta,0,gamma,0,0,0]
    #we set normalize=True because the arguments alpha, beta, gamma may not give us a normalized state
    qml.StatePrep(st, wires=range(3), normalize=True) 
    
    return qml.state()

alpha, beta, gamma = 1/np.sqrt(3), 1/np.sqrt(3), 1/np.sqrt(3),

print("The prepared state is", prep_circuit(alpha, beta, gamma))

The prepared state is [0.        +0.j 0.57735027+0.j 0.57735027+0.j 0.        +0.j
 0.57735027+0.j 0.        +0.j 0.        +0.j 0.        +0.j]


## Controlled-Unitaries

Pennylane has a function called `qml.ctrl` that takes the followig arguments:
- `op` : a Pennylane operator of the form `qml.***` (unitaries gates already defined in Pennylane) or a **quantum function**
- `control` : the list of wires that act as controls
- `control_values` : a list containing the control values assigned to each of the `control_wires`


`qml.ctrl(op, control, control_values)` act as a wrapper function that takes the parameters of `op` and the `wires` on which it acts

In [ ]:
@qml.qnode(dev)
def controlled_gate_circuit(angle):
    qml.X(0)
    #the control_value on qubit 0 is 1
    qml.ctrl(qml.RX, control=(0), control_values=(1))(angle, wires=[1])
    return qml.state()

print(controlled_gate_circuit(0))

[0.+0.j 0.+0.j 1.+0.j 0.+0.j]


If instead of a function we are given a unitary matrix $U$, we can use `qml.ControlledQubitUnitary` that takes the following arguments:
- `U` : the matrix (np.array) of the unitary $U$
- `wires` : the list composed of the control wires and the wires on which the controlled operator $U$ acts
- `control_values` : a list of the control values for the control wires

In [7]:
U = np.array([[ 0.94877869,  0.31594146], [-0.31594146,  0.94877869]])

@qml.qnode(dev3)
def circuit_controlled_unitary():
    qml.PauliX(wires = 1) #Flip to apply the controlled gate
    qml.ControlledQubitUnitary(U, wires = [0,1,2], control_values = [0,1])
    return qml.state()

print(circuit_controlled_unitary()) 

[ 0.        +0.j  0.        +0.j  0.94877869+0.j -0.31594146+0.j
  0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]


## Inverse Operations

`qml.adjoint` allows us to apply the inverse of an operator $U$

`qml.adjoint` can be used as:
1. function: `qml.adjoint(qml.RX(0.5, wires=0))`
2. wrapper: `qml.adjoint(qml.RX)(0.5, wires=0)`

However, if the argument of `qml.adjoint` is a quantum function, a wrapper must be used

In [37]:
#@qml.qnode(dev3)
def q_circ(theta, phi, omega):
    qml.RX(theta, wires=0)
    qml.RY(phi, wires=1)
    qml.RZ(omega, wires=2)

    #return qml.state()

#@qml.qnode(dev3)
def adjoint_q_circ(theta, phi, omega):
    qml.adjoint(q_circ)(theta,phi,omega)

    #return qml.state()

@qml.qnode(dev3)
def check_id(theta, phi, omega):
    q_circ(theta, phi, omega)
    adjoint_q_circ(theta, phi, omega)

    return qml.state()

#print(q_circ(np.pi,2,0))
#print(adjoint_q_circ(np.pi,2,0))
print(check_id(0,0,0))

[1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
